## Trying to implement various chain pipelines

### First Implementing Job Description Extraction pipeline

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
from pydantic import BaseModel, Field
from typing import List, Optional

class JobDescription(BaseModel):
    title: str = Field(description="The title of the job", examples=["Software Engineer", "Data Scientist"])
    company: str = Field(description="The company offering the job", examples=["Google", "Microsoft"])
    location: str = Field(description="The location of the job", examples=["San Francisco, CA", "New York, NY"])
    education: str = Field(description="The education requirements for the job", examples=["Bachelor's degree in Computer Science", "Master's degree in Data Science"])
    experience: int = Field(description="The experience requirements for the job in years", examples=[3, 5, 7])
    employment_type: str = Field(description="The type of employment", examples=["Full-time", "Part-time", "Contract", "Internship"])
    required_skills: List[str] = Field(description="The requirements of the job", examples=["3+ years of experience in software development", "Proficiency in Python and JavaScript", "Fluent in German"])
    soft_skills: List[str] = Field(description="The soft skills required for the job", examples=["Good communication skills", "Ability to work in a team"])
    responsibilities: List[str] = Field(description="The responsibilities of the job", examples=["Develop and maintain software applications", "Collaborate with cross-functional teams"])
    salary_range: Optional[str] = Field(description="The salary range for the job", examples=["$80,000 - $120,000", "1,00,000Rs - $5,00,000Rs"])

class Resume(BaseModel):
    name: str = Field(description="The full name of the person", examples=["John Doe", "Jane Smith"])
    email: str = Field(description="The email address of the person", examples=["john.doe@example.com", "jane.smith@example.com"])
    phone: str = Field(description="The phone number of the person", examples=["123-456-7890", "987-654-3210"])
    education: List[str] = Field(description="The education details of the person", examples=["Bachelor's degree in Computer Science from XYZ University", "Master's degree in Data Science from ABC University"])
    location: str = Field(description="The location of the person", examples=["San Francisco, CA", "New York, NY"])
    experience: int = Field(description="The experience details of the person in years", examples=[3, 5, 7])
    skills: List[str] = Field(description="The skills of the person", examples=["Python, JavaScript, SQL", "Machine Learning, Data Analysis, Deep Learning"])
    soft_skills: List[str] = Field(description="The soft skills of the person", examples=["Good communication skills", "Ability to work in a team"])
    certifications: Optional[List[str]] = Field(description="The certifications of the person", examples=["AWS Certified Solutions Architect", "Certified Data Scientist"])
    languages: Optional[List[str]] = Field(description="The languages known by the person", examples=["English", "Spanish", "French"])

class MatchingResponse(BaseModel):
    score: int = Field(description="The score indicating how well the resume matches the job description", ge=0, le=100, examples=[85, 90, 95])
    matched_skills: List[str] = Field(description="The list of skills that matched between the resume and the job description", examples=["Python", "JavaScript", "Communication skills"])
    skill_gaps: List[str] = Field(description="The list of skills that are required by the job description but are missing in the resume", examples=["SQL", "Machine Learning", "Experience gaps", "Leadership experience"])
    suggestions: List[str] = Field(description="The list of suggestions to improve the resume to better match the job description", examples=["Add SQL to your skills section", "Highlight your experience with machine learning in your resume"])

class ATSResponse(BaseModel):
    score: int = Field(description="The score indicating how well the resume is optimized for ATS", ge=0, le=100, examples=[80, 85, 90])
    issues: List[str] = Field(description="The list of issues that are affecting the resume's ATS optimization", examples=["Missing keywords", "Unusual formatting", "Lack of section headers"])
    suggestions: List[str] = Field(description="The list of suggestions to improve the resume's ATS optimization", examples=["Add relevant keywords from the job description", "Use standard section headers like 'Experience' and 'Education'", "Avoid using tables and graphics in your resume"])

In [126]:
JD_PARSER_PROMPT = """
You are a helpful assistant that extracts relevant information from a job description.
The information you extract will be used to analyze a resume and generate a cover letter.
You must focus on information that can be used to check how well the a resume matches a job description and to generate a cover letter that is tailored to the job description.
The information you extract should be in the form of a JSON object.
"""

RESUME_PARSER_PROMPT = """
You are a helpful assistant that extracts relevant information from a resume.
The information you extract will be used to analyze a job description and generate a cover letter.
You must focus on information that can be used to check how well the resume matches a job description and to generate a cover letter that is tailored to the job description.
You must also extract the person's name and contact, skills and experience etc.
The information you extract should be in the form of a JSON object.
"""

MATCH_ANALYZER_PROMPT = """
You are a helpful assistant that analyzes how well a resume matches a job description.
The information you analyze will be used to generate a cover letter that is tailored to the job description.
The information you analyze should be in the form of a JSON object.
The JSON object should contain a score that indicates how well the resume matches the job description
The information must include the list of skill gaps, matched skills, and experience gaps.
"""

ATS_CHECKER_PROMPT = """
You are a helpful assistant that checks how well a resume is optimized for Applicant Tracking Systems (ATS).
The information you analyze will be used to generate a cover letter that is tailored to the job description
The information you analyze should be in the form of a JSON object.
The JSON object should contain a score that indicates how well the resume is optimized for ATS.
"""

COVER_LETTER_GENERATOR_PROMPT = """
You are a helpful assistant that generates a cover letter based on a job description and a resume.
The cover letter should be tailored to the job description and should highlight the relevant skills and experience from the resume.
Make appropriate decisions about information that are missing in the resume but are relevant to the job description
The cover letter should be in the form of a well-written text that can be sent to a potential employer.
The cover letter should be formal and should follow the standard format of a cover letter, including an introduction, body, and conclusion.
For writing the date section use the format of day-th month year, for example, 1st January 2024, 10th March 2024 etc.
There should be proper subject line in the cover letter. Don't use things like RE. eg. "Subject: Application for Software Engineer position at Google"
"""

COVER_EMAIL_GENERATOR_PROMPT = """
You are a helpful assistant that generates a cover email based on a job description and a resume.
The cover email should be tailored to the job description and should highlight the relevant skills and experience from the resume.
Make appropriate decisions about information that are missing in the resume but are relevant to the job description
"""


In [28]:
import os, requests
r = requests.get(
  "https://generativelanguage.googleapis.com/v1beta/models",
  params={"key": os.environ["GOOGLE_API_KEY"]}
)

for model in r.json()["models"]:
    print(model["name"])

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025
models/gemini-embedding-001
models/gemini-embedding-2-

In [29]:
from langchain_google_genai import ChatGoogleGenerativeAI

google_api = os.getenv("GOOGLE_API_KEY")

llm = ChatGoogleGenerativeAI(
    model = "models/gemini-3-flash-preview",
    api_key = google_api
)

llm.invoke("Hello")



AIMessage(content=[{'type': 'text', 'text': 'Hello! How can I help you today?', 'extras': {'signature': 'EpcBCpQBAb4+9vumcg6KmJyz2f5+5E5fvwZlEMEOuqP+M18vUtKhJjRt2Sio7bd8kFsKYVUkpOe+T/ni841ZkwzKNAzAZ25iFbApDWg5/J+NZlCq4o9wOreMfGXvp14gPl8fUG6B9lbXnCoddTeqVUieJ5iKYdOT9NYsMLMs9ORNafj6PuUxesCfXBVIAkeFYhVly0J2wCSarA=='}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d09bb-c8bd-7c50-ae15-767ed5d7c1ee-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 32, 'total_tokens': 34, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 23}})

In [45]:
job_desc = """
**Job Title:** Senior Cloud Integration Specialist
**Department:** Technology & Innovation
**Reports To:** Director of Platform Engineering
**Location:** Austin, Texas (Hybrid - 3 days in-office)

**About the Company:**
At **NexusLogic Solutions**, we're not just riding the wave of digital transformation—we're building the surfboard. We partner with forward-thinking enterprises to streamline their operations through bespoke cloud architectures and intelligent automation. Our culture is built on curiosity, collaboration, and a healthy dose of caffeine. We were recently named one of the "Best Places to Work" by *Austin Business Journal* for the third year running.

**Position Overview:**
We are looking for a hands-on **Senior Cloud Integration Specialist** to join our growing Platform Engineering team. In this role, you will be the bridge between our development teams and cloud infrastructure, designing robust APIs and integrating third-party services. You will take ownership of our middleware solutions, ensuring data flows seamlessly and securely between our legacy systems and modern microservices.

**Key Responsibilities:**

- **Architect & Implement:** Design and develop scalable integration solutions using AWS services (API Gateway, Lambda, SQS) and integration frameworks like MuleSoft or Apache Camel.
- **API Management:** Lead the full lifecycle of API development, from design and documentation (OpenAPI/Swagger) to deployment and versioning.
- **Troubleshooting:** Act as the escalation point for complex integration issues, diagnosing bottlenecks and resolving data inconsistencies in production environments.
- **Collaboration:** Work closely with Software Engineers, Data Scientists, and Product Managers to translate business requirements into technical integration specs.
- **Mentorship:** Guide junior developers on best practices for integration patterns, security protocols (OAuth 2.0, JWT), and code maintainability.

**Qualifications:**

- 5+ years of experience in software development or integration engineering.
- Deep expertise in at least one major cloud provider (AWS preferred).
- Strong proficiency in Java, Python, or Node.js.
- Experience with message queues and streaming platforms (Kafka, RabbitMQ).
- Bachelor's degree in Computer Science or equivalent practical experience.

**Why Join NexusLogic?**

- **Compensation:** Competitive salary range: **$145,000 - $175,000** per year, plus performance-based bonus.
- **Benefits:** Comprehensive medical, dental, and vision coverage; 401(k) with 5% company match.
- **Work-Life Balance:** Generous PTO policy, paid parental leave, and flexible working hours.
- **Perks:** Weekly team lunches, annual innovation retreat, and a $1,500 annual stipend for professional development.

---

**How to Apply:**
Please send your resume and a brief cover letter to **Hiring Manager, Alex Chen**, at **careers@nexuslogic-solutions.io** with the subject line: "Senior Cloud Integration Specialist Application"."""

In [44]:
from langchain.messages import HumanMessage
from langchain.agents import create_agent

def parse_job_description(llm, job_desc: str) -> JobDescription:
    agent = create_agent(
        model = llm,
        system_prompt = JD_PARSER_PROMPT,
        response_format = JobDescription
    )
    query = HumanMessage(content = job_desc)
    response = agent.invoke({
        "messages": [query]
    })
    return response['structured_response']

In [46]:
res = parse_job_description(llm, job_desc)
#.model_dump_json(indent=2)

print(res)

title='Senior Cloud Integration Specialist' company='NexusLogic Solutions' location='Austin, Texas (Hybrid - 3 days in-office)' education="Bachelor's degree in Computer Science or equivalent practical experience" experience=5 employment_type='Full-time' required_skills=['AWS', 'API Gateway', 'Lambda', 'SQS', 'MuleSoft', 'Apache Camel', 'Java', 'Python', 'Node.js', 'Kafka', 'RabbitMQ', 'OpenAPI', 'Swagger', 'OAuth 2.0', 'JWT'] soft_skills=['Curiosity', 'Collaboration', 'Mentorship', 'Problem-solving', 'Communication'] responsibilities=['Design and develop scalable integration solutions using AWS services and integration frameworks', 'Lead the full lifecycle of API development from design to deployment and versioning', 'Act as the escalation point for complex integration issues and data inconsistencies', 'Collaborate with Software Engineers, Data Scientists, and Product Managers', 'Guide junior developers on integration patterns, security protocols, and code maintainability'] salary_rang

In [47]:
print(res.model_dump_json(indent=2))

{
  "title": "Senior Cloud Integration Specialist",
  "company": "NexusLogic Solutions",
  "location": "Austin, Texas (Hybrid - 3 days in-office)",
  "education": "Bachelor's degree in Computer Science or equivalent practical experience",
  "experience": 5,
  "employment_type": "Full-time",
  "required_skills": [
    "AWS",
    "API Gateway",
    "Lambda",
    "SQS",
    "MuleSoft",
    "Apache Camel",
    "Java",
    "Python",
    "Node.js",
    "Kafka",
    "RabbitMQ",
    "OpenAPI",
    "Swagger",
    "OAuth 2.0",
    "JWT"
  ],
  "soft_skills": [
    "Curiosity",
    "Collaboration",
    "Mentorship",
    "Problem-solving",
    "Communication"
  ],
  "responsibilities": [
    "Design and develop scalable integration solutions using AWS services and integration frameworks",
    "Lead the full lifecycle of API development from design to deployment and versioning",
    "Act as the escalation point for complex integration issues and data inconsistencies",
    "Collaborate with Software

## Resume Parsing

1. Read PDF and convert to string
2. Parse the resume

In [9]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "resume.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()
print(docs)

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2026-03-16T20:10:34+05:30', 'author': 'MIDHUN U', 'moddate': '2026-03-16T20:10:34+05:30', 'source': 'resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='AMAL VARGHESE \nLinkedIn | GitHub | Portfolio amalvarghesealiyattukudy.mec@gmail.com \nDOB – 15/05/2004 +91 9207506741 \nGovt. Model Engineering College, Kochi \nSKILLS & INTERESTS \n• Technical Skills: Python, Java, MySQL, MongoDB, Data Structures and Algorithms, OOPS, React, React Native \n• Soft Skills:  Communication, Problem Solving, Teamwork, Adaptability, Critical Thinking, Attention to Detail, Research, \n                                                 Time Management, Team Collaboration                                                                  \n• Interests:  Machine Learning, Web Development, Artificial Intelligence \nEDUCATION \n• Govt. Model Engineering College

In [26]:
print(docs[0].page_content)

AMAL VARGHESE 
LinkedIn | GitHub | Portfolio amalvarghesealiyattukudy.mec@gmail.com 
DOB – 15/05/2004 +91 9207506741 
Govt. Model Engineering College, Kochi 
SKILLS & INTERESTS 
• Technical Skills: Python, Java, MySQL, MongoDB, Data Structures and Algorithms, OOPS, React, React Native 
• Soft Skills:  Communication, Problem Solving, Teamwork, Adaptability, Critical Thinking, Attention to Detail, Research, 
                                                 Time Management, Team Collaboration                                                                  
• Interests:  Machine Learning, Web Development, Artificial Intelligence 
EDUCATION 
• Govt. Model Engineering College 9.37 | 2027 
KTU, B.Tech in Computer Science Engineering  
• Greenvalley Public School 94.4% | 2022 
CBSE, 12th     
• St. Thomas Public School 95% | 2020 
CBSE, 10th  
WORK EXPERIENCE 
• Wrench Solutions 1 Month, Ongoing 
Intern 
Technologies Used: Python, Pandas, NumPy, Matplotlib, PyTorch 
Currently working as a Mac

In [18]:
RESUME_PARSER_PROMPT = """
You are a helpful assistant that extracts relevant information from a resume.
The information you extract will be used to analyze a job description and generate a cover letter.
You must focus on information that can be used to check how well the resume matches a job description and to generate a cover letter that is tailored to the job description.
You must also extract the person's name and contact, skills and experience etc.
The information you extract should be in the form of a JSON object.
"""

class Resume(BaseModel):
    name: str = Field(description="The full name of the person", examples=["John Doe", "Jane Smith"])
    email: str = Field(description="The email address of the person", examples=["john.doe@example.com", "jane.smith@example.com"])
    phone: str = Field(description="The phone number of the person", examples=["123-456-7890", "987-654-3210"])
    education: List[str] = Field(description="The education details of the person", examples=["Bachelor's degree in Computer Science from XYZ University", "Master's degree in Data Science from ABC University"])
    location: str = Field(description="The location of the person", examples=["San Francisco, CA", "New York, NY"])
    experience: int = Field(description="The experience details of the person in years", examples=[3, 5, 7])
    skills: List[str] = Field(description="The skills of the person", examples=["Python, JavaScript, SQL", "Machine Learning, Data Analysis, Deep Learning"])
    soft_skills: List[str] = Field(description="The soft skills of the person", examples=["Good communication skills", "Ability to work in a team"])
    certifications: Optional[List[str]] = Field(description="The certifications of the person", examples=["AWS Certified Solutions Architect", "Certified Data Scientist"])
    languages: Optional[List[str]] = Field(description="The languages known by the person", examples=["English", "Spanish", "French"])

In [52]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

def parse_resume(llm: ChatGoogleGenerativeAI, resume_text: str) -> Resume:
    agent = create_agent(
        model = llm,
        system_prompt = RESUME_PARSER_PROMPT,
        response_format = Resume
    )
    query = HumanMessage(content = resume_text)
    response = agent.invoke({
        "messages": [query]
    })
    return response['structured_response']


In [40]:
resume = parse_resume(llm, docs[0].page_content)

In [41]:
print(resume.model_dump_json(indent=2))

{
  "name": "AMAL VARGHESE",
  "email": "amalvarghesealiyattukudy.mec@gmail.com",
  "phone": "+91 9207506741",
  "education": [
    "Govt. Model Engineering College: B.Tech in Computer Science Engineering (Expected 2027)",
    "Greenvalley Public School: CBSE 12th (2022)",
    "St. Thomas Public School: CBSE 10th (2020)"
  ],
  "location": "Kochi",
  "experience": 0,
  "skills": [
    "Python",
    "Java",
    "MySQL",
    "MongoDB",
    "Data Structures and Algorithms",
    "OOPS",
    "React",
    "React Native",
    "Pandas",
    "NumPy",
    "Matplotlib",
    "PyTorch",
    "ONNX",
    "FastAPI",
    "Supabase",
    "SQL",
    "XGBoost",
    "Next.js",
    "LangChain",
    "BrightData"
  ],
  "soft_skills": [
    "Communication",
    "Problem Solving",
    "Teamwork",
    "Adaptability",
    "Critical Thinking",
    "Attention to Detail",
    "Research",
    "Time Management",
    "Team Collaboration"
  ],
  "certifications": [
    "Programming, Data Structures and Algorithms using

## Agent for Analyzing resume with Job description

In [53]:
def analyze_match(llm, job_desc: JobDescription, resume: Resume) -> MatchingResponse:
    """
    This agent will take the job description and resume in json format and analyze how well the resume matches the job description.
    
    Args:
        llm: The language model instance to use for analysis.
        job_desc (JobDescription)
        resume (Resume)
    Return:
        MatchingResponse: A JSON object containing the match percentage and a list of matching and non-matching skills, experience, education etc.
    """
    agent = create_agent(
        model = llm,
        system_prompt = MATCH_ANALYZER_PROMPT,
        response_format = MatchingResponse
    )

    query1 = HumanMessage(content = job_desc.model_dump_json(indent=2))
    query2 = HumanMessage(content = resume.model_dump_json(indent=2))
    response = agent.invoke({
        "messages": [query1, query2]
    })

    return response['structured_response']

In [54]:
response = analyze_match(llm, res, resume)

In [55]:
print(response.model_dump_json(indent=2))

{
  "score": 15,
  "matched_skills": [
    "Java",
    "Python",
    "Communication",
    "Problem-solving",
    "Collaboration"
  ],
  "skill_gaps": [
    "AWS",
    "API Gateway",
    "Lambda",
    "SQS",
    "MuleSoft",
    "Apache Camel",
    "Node.js",
    "Kafka",
    "RabbitMQ",
    "OpenAPI",
    "Swagger",
    "OAuth 2.0",
    "JWT",
    "5 years of professional experience",
    "Mentorship experience"
  ],
  "suggestions": [
    "Obtain AWS certifications such as Certified Solutions Architect or Developer to bridge the cloud infrastructure gap.",
    "Gain hands-on experience with enterprise integration tools like MuleSoft or Apache Camel.",
    "Build and showcase projects involving asynchronous messaging systems like Kafka or RabbitMQ.",
    "Learn and implement industry-standard API security protocols such as OAuth 2.0 and JWT in personal projects.",
    "Familiarize yourself with API documentation standards like OpenAPI and Swagger.",
    "Apply for internship or entry-le

## Agent For checking ATS Friendliness

In [116]:
def check_ats(llm, resume: str) -> ATSResponse:
    """
    This agent will take the resume in string format and analyze how well it is optimized for ATS.
    Args:
        llm: The language model instance to use for analysis.
        resume (string): The resume text to analyze.
    Return:
        ATSResponse: a json response containing ATS score and suggestions
    """
    agent = create_agent(
        model = llm,
        system_prompt = ATS_CHECKER_PROMPT,
        response_format = ATSResponse
    )
    query = HumanMessage(content = resume)

    response = agent.invoke({
        "messages": [query]
    })

    return response['structured_response']

In [63]:
resume_text = docs[0].page_content
ats_report = check_ats(llm, resume_text)

In [64]:
print(ats_report.model_dump_json(indent=2))

{
  "score": 85,
  "issues": [
    "Inclusion of personal details like Date of Birth which is unnecessary for ATS and can trigger bias filters.",
    "The inclusion of a References section takes up valuable space and is typically requested later in the hiring process.",
    "Professional experience and projects use '1 Month' or '3 Months' instead of specific start and end dates (e.g., June 2024 - Present).",
    "The Hobbies section contains non-professional information that does not contribute to keyword matching.",
    "Soft skills are listed in a block, which can be less effective than integrating them into achievement bullets."
  ],
  "suggestions": [
    "Remove the Date of Birth and Hobbies sections to keep the resume strictly professional.",
    "Replace 'References available upon request' or specific contact details with more detailed project descriptions or technical achievements.",
    "Use standard date formats (Month Year - Month Year) for all work experience and project du

### Agent to build cover letter generator

In [121]:
def generate_cover_letter(llm, job_desc: JobDescription, resume: Resume, match_response: MatchingResponse, tone: str = "Professional") -> str:
    """
    This agent will take the job description and resume in json format and the match response and generate a cover letter that is tailored to the job description and highlights the skills and experience that match the job description.
    Args:
        llm: The language model instance to use for analysis.
        job_desc (JobDescription)
        resume (Resume)
        match_response (MatchingResponse)
    Return:
        str: A cover letter that is tailored to the job description and highlights the skills and experience that match the job description.
    """

    agent = create_agent(
        model = llm,
        system_prompt = COVER_LETTER_GENERATOR_PROMPT
    )

    query1 = HumanMessage(content=job_desc.model_dump_json(indent=2))
    query2 = HumanMessage(content = resume.model_dump_json(indent=2))
    query3 = HumanMessage(content = match_response.model_dump_json(indent=2))
    query4 = HumanMessage(content = f"The tone of the cover letter should be {tone}")

    response = agent.invoke({
        "messages": [query1, query2, query3, query4]
    })

    return response["messages"][4].content[0]["text"]

In [122]:
letter = generate_cover_letter(llm, res, resume, response)

In [123]:
print(letter)

Amal Varghese
+91 9207506741
amalvarghesealiyattukudy.mec@gmail.com
Kochi, Kerala

24th May 2024

Hiring Manager
NexusLogic Solutions
Austin, Texas

Subject: Application for Senior Cloud Integration Specialist position at NexusLogic Solutions

Dear Hiring Manager,

I am writing to express my enthusiastic interest in the Senior Cloud Integration Specialist position at NexusLogic Solutions, as advertised. With a robust background in software engineering and a deep specialization in building scalable, interconnected systems, I am confident that my technical expertise in Java, Python, and cloud-native architectures aligns perfectly with the goals of your integration team.

Throughout my career, I have focused on designing high-performance backend systems and APIs. My proficiency in Java and Python has allowed me to build resilient services, and I have extensive experience working with modern frameworks to streamline data flow across complex environments. At my previous engagements, I have 

In [ ]:
def generate_cover_email(llm, job_desc: JobDescription, resume: Resume, match_response: MatchingResponse) -> str:
    """
    This agent will take the job description and resume in json format and the match response and generate a cover letter that is tailored to the job description and highlights the skills and experience that match the job description.
    Args:
        llm: The language model instance to use for analysis.
        job_desc (JobDescription)
        resume (Resume)
        match_response (MatchingResponse)
    Return:
        str: A cover email that is tailored to the job description and highlights the skills and experience that match the job description.
    """
    agent = create_agent(
        model = llm,
        system_prompt = COVER_EMAIL_GENERATOR_PROMPT
    )

    query1 = HumanMessage(content=job_desc.model_dump_json(indent=2))
    query2 = HumanMessage(content = resume.model_dump_json(indent=2))
    query3 = HumanMessage(content = match_response.model_dump_json(indent=2))

    response = agent.invoke({
        "messages": [query1, query2, query3]
    })

    return response["messages"][3].content[0]["text"]

In [ ]:
email = generate_cover_email(llm, res, resume, response)

In [129]:
print(email)

Subject: Application for Senior Cloud Integration Specialist - Amal Varghese

Dear Hiring Manager,

I am writing to express my enthusiastic interest in the Senior Cloud Integration Specialist position at NexusLogic Solutions, as advertised. With a strong technical foundation in computer science and a passion for building scalable, high-performance integration architectures, I am eager to contribute to your team’s success in Austin.

Throughout my academic and project-based experience, I have developed a deep proficiency in Python and Java—two of the core languages required for this role. My background in developing robust backends using FastAPI and managing complex data environments with SQL and MongoDB aligns with NexusLogic’s focus on sophisticated API development. I have a proven track record of solving complex problems, from optimizing data structures to implementing AI-driven solutions using LangChain and PyTorch.

Key highlights of my qualifications include:
*   **API Development